In [ ]:
from datetime import datetime
import os 

In [ ]:
data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-12-13_preparing_cohort_genotype_files_for_regenie_input"

results = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

scratch =  "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

#!mkdir {scratch}

In [ ]:
!bash -lc 'rm -f acaf_pmerge_list.txt; for c in {1..22} X Y; do echo acaf_threshold.chr${c} >> acaf_pmerge_list.txt; done'

In [ ]:
%%bash

pgen_dir_in="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-12-13_preparing_cohort_genotype_files_for_regenie_input/acaf_pgen_files/pgen"

pgen_dir_out="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

for chr in {1..22}; do
  plink2 \
    --pfile ${pgen_dir_in}/acaf_threshold.chr${chr} \
    --set-missing-var-ids @:# \
    --maf 0.01 \
    --mac 100 \
    --geno 0.1 \
    --hwe 1e-15 \
    --mind 0.1 \
    --make-pgen \
    --out ${pgen_dir_out}/acaf_threshold.chr${chr}.qc
done


In [ ]:
%%bash


pgen_dir_data="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-12-13_preparing_cohort_genotype_files_for_regenie_input/acaf_pgen_files/pgen"

pgen_dir_scr="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"



for chr in {1..22}; do
  plink2 \
    --pfile ${pgen_dir_scr}/acaf_threshold.chr${chr}.qc \
    --indep-pairwise 1000 100 0.9 \
    --out ${pgen_dir_scr}/acaf_threshold.chr${chr}.ld_prune
done


In [ ]:
pgen_dir = "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"  # where the .qc pfiles actually live
scratch_dir = "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

chroms = range(1, 23)

with open(f"{scratch_dir}/acaf_qc_pmerge_list.txt", "w") as f:
    for c in chroms:
        # This must match your plink2 --out prefix when you did per-chr QC
        f.write(f"{pgen_dir}/acaf_threshold.chr{c}.qc\n")


In [ ]:
%%bash

pgen_dir_scr="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"


plink2 \
  --pmerge-list ${pgen_dir_scr}/acaf_qc_pmerge_list.txt \
  --set-missing-var-ids @:# \
  --make-pgen \
  --out ${pgen_dir_scr}/acaf_threshold.allchr.qc_merge


In [ ]:
%%bash


pgen_dir_scr="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-13_preparing_cohort_genotype_files_for_regenie_input"

cat ${pgen_dir_scr}/acaf_threshold.chr*.ld_prune.prune.in > ${pgen_dir_scr}/acaf_threshold.allchr.ld_prune.prune.in


In [ ]:
%%bash

pgen_dir_scr="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-13_preparing_cohort_genotype_files_for_regenie_input"
pgen_dir_res="/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"


plink2 \
  --pfile ${pgen_dir_scr}/acaf_threshold.allchr.qc_merge \
  --extract acaf_threshold.allchr.ld_prune.prune.in \
  --make-pgen \
  --out ${pgen_dir_res}/acaf_threshold.step1_snps


In [ ]:
%%bash

pgen_dir_res="/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"


awk 'NR>1 && $1 ~ /^[0-9]+$/ && $1>=1 && $1<=22 {count[$1]++} \
     END {for (c in count) print c, count[c]}' \
  ${pgen_dir_res}/acaf_threshold.step1_snps.pvar \
  | sort -k1,1V > snp_counts_1_22.txt


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("snp_counts_1_22.txt", sep=r"\s+", header=None, names=["chrom", "count"])

# ensure numeric and sorted
df["chrom"] = df["chrom"].astype(int)
df = df.sort_values("chrom")

plt.figure(figsize=(10, 4))
plt.bar(df["chrom"].astype(str), df["count"])
plt.xlabel("Chromosome")
plt.ylabel("Number of variants")
plt.title("Number of variants per chromosome (1–22)")
plt.tight_layout()
plt.show()

